# 05 — Phase 3a: image CO-REGISTRATION

Aligns every shoreline scene (Series A ∪ B, 880 unique scenes) to ONE clear
Sentinel-2 reference, then shifts each shoreline vector by its scene's offset.
Logic lives in `src/coreg.py` (method + citations in its docstring).

**Run in order. Stop at each ✅ CHECK and send me the printed output.**
1. Setup → 2. Self-test (offline) → 3. Chips → 4. Reference (automatic) →
5. Reference-vs-itself check → 6. Smoke test (8 scenes, scratch folder) →
7. Full run (resumable) → 8. QC → 9. Apply shifts to shorelines + push.

## 1. Setup — clone-or-pull, GEE, Drive, imports

In [ ]:
import os
REPO_DIR = '/content/Shoreline-Dynamics'
if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull -q
else:
    !git clone https://github.com/SKPrince1911/Shoreline-Dynamics.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements.txt
!git log --oneline -1

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='shoreline-analysis')
print('GEE initialized:', ee.String('ok').getInfo())
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys, importlib; sys.path.append('.')
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from src import config, data, extract, coreg
importlib.reload(config); importlib.reload(data); importlib.reload(extract); importlib.reload(coreg)
import inspect; print('coreg.py loaded from:', inspect.getfile(coreg))

## 2. Self-test (offline, no GEE)
Known synthetic shifts (up to ±2.5 px) must be recovered.
✅ **CHECK:** mean error < 1.5 m and max < 3 m (on a 30 m grid).

In [ ]:
st = coreg.selftest()
print(st['err_m'].describe().round(2))
print('failed cases:', st['err_m'].isna().sum())

## 3. Chips on the land side of the baseline
✅ **CHECK:** about 62 chips, all on LAND (east of the beach), along the whole coast.

In [ ]:
chips = coreg.build_chips()
base = gpd.read_file(extract.BASELINE_PATH).to_crs(config.METRIC_CRS)
zone = extract.load_search_zone()
ax = chips.plot(facecolor='none', edgecolor='tab:red', figsize=(6, 10))
base.plot(ax=ax, color='k', lw=1)
if zone is not None: gpd.GeoSeries([zone], crs=config.METRIC_CRS).plot(ax=ax, color='tab:blue', alpha=0.3)
ax.set_title(f'{len(chips)} co-registration chips (red), baseline (black), search zone (blue)')
plt.show()

## 4. Reference scene (picked automatically)
Clearest dry-season Sentinel-2 scene 2022–2026 whose chips are ≥ 85 % usable.
Recorded in `outputs/coreg/coreg_reference.json` (+ Drive) and reused on resume.
✅ **CHECK:** send me the chosen `image_id` and usable fraction.

In [ ]:
scene_list = extract.get_scene_list_dense()
reference = coreg.load_or_select_reference(chips, scene_list=scene_list)
print('REFERENCE:', reference['image_id'], '| usable chips:', round(reference['usable_frac'], 3))

## 5. Reference against itself
✅ **CHECK:** dx, dy ≈ 0 (|value| < 0.5 m) and status `ok`.

In [ ]:
rec, per_chip = coreg.coregister_scene(reference['image_id'], 'S2', chips, reference)
print({k: rec[k] for k in ['status', 'n_chips_used', 'dx_m', 'dy_m', 'se_dx_m', 'se_dy_m']})

## 6. Smoke test — 10 scenes, written to a SCRATCH folder (never committed)
Two scenes per sensor family from different decades.
✅ **CHECK:** status mostly `ok`; shifts typically within ~±15 m; seconds per scene.

In [ ]:
scenes = coreg.scenes_to_coregister()
print(len(scenes), 'unique scenes'); print(scenes['sensor'].value_counts().to_dict())
pick = pd.concat([g.iloc[[0, len(g) // 2]] for _, g in scenes.groupby('sensor')])
smoke = coreg.run_coregistration(pick, chips, reference, out_dir='/content/smoke')
smoke[['sensor', 'acq_datetime_utc', 'status', 'n_chips_used', 'dx_m', 'dy_m',
       'se_dx_m', 'se_dy_m', 'spread_m', 'median_r']].round(2)

In [ ]:
# Per-chip detail for one smoke scene (are chip shifts consistent along the coast?)
row = pick.iloc[0]
rec, pc = coreg.coregister_scene(row.image_id, row.sensor, chips, reference)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].scatter(pc['along_km'], pc['dx_m'], label='dx'); ax[0].scatter(pc['along_km'], pc['dy_m'], label='dy')
ax[0].axhline(rec['dx_m'], c='C0', ls='--'); ax[0].axhline(rec['dy_m'], c='C1', ls='--')
ax[0].set_xlabel('along-coast km (N→S)'); ax[0].set_ylabel('shift (m)'); ax[0].legend()
ax[1].hist(pc['r'], bins=15); ax[1].set_xlabel('chip correlation r')
plt.suptitle(f"{row.sensor} {str(row.acq_datetime_utc)[:10]}  status={rec['status']}"); plt.show()

## 7. FULL RUN (resumable) — ~880 scenes
Only after I confirm the smoke test. Checkpoints every 10 scenes to
`outputs/coreg/coreg_shifts.csv` **and** Drive. If Colab disconnects, re-run
cells 1, 3, 4 and this cell — finished scenes are skipped.

In [ ]:
shifts = coreg.run_coregistration(scenes, chips, reference)
shifts['status'].value_counts()

## 8. QC
✅ **CHECK:** send me these tables and plots.

In [ ]:
shifts = pd.read_csv(os.path.join(coreg.DRIVE_COREG_DIR, coreg.SHIFTS_CSV))
print(pd.crosstab(shifts['sensor'], shifts['status']))
ok = shifts[shifts['status'] == 'ok'].copy()
ok['shift_m'] = np.hypot(ok['dx_m'], ok['dy_m'])
print(ok.groupby('sensor')[['dx_m', 'dy_m', 'shift_m', 'se_dx_m', 'spread_m', 'median_r']]
        .median().round(2))
print('\nscenes with |shift| > 30 m:', (ok['shift_m'] > 30).sum())
print(ok.loc[ok['shift_m'] > 30, ['image_id', 'sensor', 'dx_m', 'dy_m', 'n_chips_used']].head(20))
t = pd.to_datetime(ok['acq_datetime_utc'], utc=True)
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
for s, g in ok.groupby('sensor'):
    tt = pd.to_datetime(g['acq_datetime_utc'], utc=True)
    ax[0].scatter(tt, g['dx_m'], s=8, label=s); ax[1].scatter(tt, g['dy_m'], s=8, label=s)
ax[0].set_ylabel('dx (m, east)'); ax[1].set_ylabel('dy (m, north)'); ax[0].legend(ncol=5)
plt.suptitle('Co-registration shifts by scene'); plt.show()

## 9. Apply shifts to the shoreline vectors + save + push
Writes `sds_scenes_coreg.geojson` (A) and `sds_scenes_B_coreg.geojson` (B) next
to the originals (originals untouched). Scenes not `ok` keep their geometry and
are flagged in `coreg_status`.

In [ ]:
A = gpd.read_file(extract.SDS_SCENES_PATH)
B = gpd.read_file(os.path.join(extract.SHORELINE_DIR, 'sds_scenes_B.geojson'))
A2 = coreg.apply_shifts(A, shifts); B2 = coreg.apply_shifts(B, shifts)
for gdf, name in ((A2, 'sds_scenes_coreg.geojson'), (B2, 'sds_scenes_B_coreg.geojson')):
    for p in extract._shoreline_out_paths(name):
        os.makedirs(os.path.dirname(p), exist_ok=True); gdf.to_file(p, driver='GeoJSON'); print('wrote', p)
print(A2['coreg_status'].value_counts().to_dict(), B2['coreg_status'].value_counts().to_dict())
import shutil
os.makedirs(coreg.COREG_DIR, exist_ok=True)
for f in (coreg.SHIFTS_CSV, coreg.REFERENCE_JSON, coreg.CHIPS_GEOJSON):
    src = os.path.join(coreg.DRIVE_COREG_DIR, f)
    if os.path.exists(src): shutil.copy(src, os.path.join(coreg.COREG_DIR, f))
print(extract.push_outputs_to_github('Phase 3a: co-registration shifts + co-registered shorelines'))